# Tutorial: Using libcint with GBasis

This tutorial demonstrates how to use the `libcint` C backend in GBasis for fast integral evaluation.

**What you will learn:**
- How to set up a molecular system using GBasis
- How to compute integrals using the `libcint` C backend via `CBasis`
- How to use 1-electron, 2-electron, gradient, and 3-center integrals

**Reference:** Li, Q., & Sun, Q. (2024). *J. Chem. Phys.* — libcint: An efficient general integral library for Gaussian basis functions.

**Issue #229:** This tutorial is part of the GSoC 2026 project to integrate libcint into GBasis.

## Installation

```bash
pip install qc-gbasis
```

libcint is automatically compiled and bundled during installation on Linux and macOS.

In [ ]:
import numpy as np

from gbasis.parsers import make_contractions, parse_nwchem
from gbasis.integrals.libcint import CBasis, ELEMENTS

print('GBasis imported successfully!')
print(f'libcint is available: True')

## Step 1: Define a Molecular System

We use the H2 molecule as our example — same as Figure 1 in the libcint paper.

Coordinates are in **Bohr** (atomic units).

In [ ]:
# H2 molecule — bond length 1.4 Bohr
atsyms = ['H', 'H']
atcoords = np.array([
    [0.0, 0.0, -0.7],   # H1
    [0.0, 0.0,  0.7],   # H2
])

print(f'Atoms: {atsyms}')
print(f'Coordinates (Bohr):\n{atcoords}')

## Step 2: Load Basis Set

We use the STO-6G minimal basis set.

In [ ]:
# Load STO-6G basis set
basis_dict = parse_nwchem('../../tests/data_sto6g.nwchem')

# Create GBasis contractions
py_basis = make_contractions(basis_dict, atsyms, atcoords, coord_types='spherical')

print(f'Basis set loaded: STO-6G')
print(f'Number of shells: {len(py_basis)}')

## Step 3: Create CBasis Object

`CBasis` is the main interface to the `libcint` C backend.
It wraps the `atm`, `bas`, `env` arrays required by libcint.

In [ ]:
# Create CBasis — libcint interface
cb = CBasis(py_basis, atsyms, atcoords, coord_type='spherical')

print(f'CBasis created!')
print(f'Number of atoms: {cb.natm}')
print(f'Number of shells: {cb.nbas}')
print(f'Number of basis functions: {cb.nbfn}')

## Step 4: 1-Electron Integrals

### Overlap Integral
The overlap matrix S measures how much two basis functions overlap.

In [ ]:
# Overlap integral
S = cb.overlap_integral()
print(f'Overlap matrix shape: {S.shape}')
print(f'Overlap matrix:\n{S}')
print(f'\nDiagonal (self-overlap = 1): {np.diag(S)}')

### Kinetic Energy Integral

In [ ]:
# Kinetic energy integral
T = cb.kinetic_energy_integral()
print(f'Kinetic energy matrix shape: {T.shape}')
print(f'Kinetic energy matrix:\n{T}')

### Nuclear Attraction Integral

In [ ]:
# Nuclear attraction integral
V = cb.nuclear_attraction_integral()
print(f'Nuclear attraction matrix shape: {V.shape}')
print(f'Nuclear attraction matrix:\n{V}')

### Core Hamiltonian

H_core = T + V

In [ ]:
H_core = T + V
print(f'Core Hamiltonian:\n{H_core}')

## Step 5: Momentum and Moment Integrals

In [ ]:
# Momentum integral — shape (nbfn, nbfn, 3) — complex
origin = np.zeros(3)
P = cb.momentum_integral(origin=origin)
print(f'Momentum integral shape: {P.shape}')
print(f'Momentum integral (x-component):\n{P[:,:,0]}')

In [ ]:
# Moment integral — dipole (order 1)
orders = np.array([[1,0,0], [0,1,0], [0,0,1]])  # x, y, z dipole
M = cb.moment_integral(orders, origin=origin)
print(f'Dipole moment integral shape: {M.shape}')
print(f'x-dipole:\n{M[:,:,0]}')
print(f'z-dipole:\n{M[:,:,2]}')

## Step 6: 2-Electron Integrals (ERI)

In [ ]:
# Electron Repulsion Integrals (ERI)
ERI = cb.electron_repulsion_integral()
print(f'ERI shape: {ERI.shape}')  # (nbfn, nbfn, nbfn, nbfn)
print(f'ERI[0,0,0,0] = {ERI[0,0,0,0]:.6f}')

## Step 7: Gradient Integrals (PR 7)

In [ ]:
# Overlap gradient
dS = cb.overlap_gradient_integral()
print(f'Overlap gradient shape: {dS.shape}')  # (nbfn, nbfn, 3)

# Kinetic gradient
dT = cb.kinetic_gradient_integral()
print(f'Kinetic gradient shape: {dT.shape}')

# Nuclear gradient
dV = cb.nuclear_gradient_integral()
print(f'Nuclear gradient shape: {dV.shape}')

## Step 8: 3-Center Integrals (PR 8)

In [ ]:
# 3-center 2-electron repulsion integrals
int3c = cb.three_center_repulsion_integral()
print(f'3-center integral shape: {int3c.shape}')  # (nbfn, nbfn, nbfn)
print(f'int3c[0,0,0] = {int3c[0,0,0]:.6f}')

## Step 9: Verify Against GBasis Python

We verify that libcint results match GBasis's pure Python implementation.

In [ ]:
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
import numpy.testing as npt

# GBasis Python results
S_py = overlap_integral(py_basis, screen_basis=False)
T_py = kinetic_energy_integral(py_basis, screen_basis=False)

# Compare
npt.assert_allclose(S, S_py, atol=1e-6)
npt.assert_allclose(T, T_py, atol=1e-6)

print('Overlap: libcint matches GBasis Python ✓')
print('Kinetic: libcint matches GBasis Python ✓')
print(f'Max overlap difference: {np.max(np.abs(S - S_py)):.2e}')
print(f'Max kinetic difference: {np.max(np.abs(T - T_py)):.2e}')

## Summary

In this tutorial, we demonstrated:

| Integral | Method | Shape |
|----------|--------|-------|
| Overlap | `cb.overlap_integral()` | (nbfn, nbfn) |
| Kinetic | `cb.kinetic_energy_integral()` | (nbfn, nbfn) |
| Nuclear | `cb.nuclear_attraction_integral()` | (nbfn, nbfn) |
| Momentum | `cb.momentum_integral(origin)` | (nbfn, nbfn, 3) |
| Moment | `cb.moment_integral(orders, origin)` | (nbfn, nbfn, N) |
| ERI | `cb.electron_repulsion_integral()` | (nbfn, nbfn, nbfn, nbfn) |
| Overlap gradient | `cb.overlap_gradient_integral()` | (nbfn, nbfn, 3) |
| 3-center | `cb.three_center_repulsion_integral()` | (nbfn, nbfn, nbfn) |

All results are verified to match GBasis's pure Python implementation to within 1e-6 tolerance.